<a href="https://colab.research.google.com/github/shibanidsai/MLOPS/blob/main/Model_Monitoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install alibi_detect library

In [1]:
import numpy as np
np.__version__

'2.0.2'

In [2]:
!pip install alibi alibi_detect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of thinc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of thinc to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of spacy[lookups] to determine which version is compatible with other requirements. This could take a while.
INFO: p

In [1]:
import alibi
from alibi_detect.cd import ChiSquareDrift, TabularDrift
from alibi_detect.saving import save_detector, load_detector

In [2]:
from google.colab import files
uploaded = files.upload()

Saving cars.csv to cars.csv


In [4]:

import pandas as pd
cars_df = pd.read_csv('cars.csv')


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn

In [6]:
cars_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1038 entries, 0 to 1037
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Location      1038 non-null   object 
 1   Fuel_Type     1038 non-null   object 
 2   Transmission  1038 non-null   object 
 3   Owner_Type    1038 non-null   object 
 4   Seats         1037 non-null   float64
 5   Price         1038 non-null   float64
 6   age           1038 non-null   int64  
 7   KM_Driven     1038 non-null   int64  
 8   make          1038 non-null   object 
 9   mileage       1038 non-null   float64
 10  engine        1038 non-null   int64  
 11  power         1038 non-null   float64
dtypes: float64(4), int64(3), object(5)
memory usage: 97.4+ KB


In [7]:
x_features = list(cars_df.columns)

In [9]:
x_features

['Location',
 'Fuel_Type',
 'Transmission',
 'Owner_Type',
 'Seats',
 'Price',
 'age',
 'KM_Driven',
 'make',
 'mileage',
 'engine',
 'power']

#### Specify the index of the columns which are categorical feautures

In [10]:
cat_vars = [0, 1, 2, 3, 8]

In [11]:
X = cars_df[x_features]
y = cars_df.Price

### Split the dataset into two sets

**Note**: In this exampls, data is split to create train and production datasets. This is done only for the lab session. In real world, the production data will come from the inference stystem.

In [12]:
from sklearn.model_selection import train_test_split

In [13]:
X_train, X_prod, y_train, y_prod = train_test_split(X,
                                                    y,
                                                    train_size = 0.9,
                                                    random_state = 23)

In [14]:
categories_per_feature = {f: None for f in cat_vars}

In [15]:
categories_per_feature

{0: None, 1: None, 2: None, 3: None, 8: None}

### Measure the drift

In [16]:
cd = TabularDrift(X_train.values,
                  p_val=.05,
                  categories_per_feature=categories_per_feature)

In [17]:
filepath = 'carsdrift'  # change to directory where detector is saved
save_detector(cd, filepath, legacy = True)

In [18]:
cd = load_detector(filepath)

In [19]:
preds = cd.predict(X_prod.to_numpy())

### Printing the test results

- KS test for the numerical features
- chi-squared test for the categorical features

In [20]:
for f in range(cd.n_features):
    stat = 'Chi2' if f in list(categories_per_feature.keys()) else 'K-S'
    fname = x_features[f]
    stat_val, p_val = preds['data']['distance'][f], preds['data']['p_val'][f]
    print(f'{fname} -- {stat} {stat_val:.3f} -- p-value {p_val:.3f}')

Location -- Chi2 8.221 -- p-value 0.607
Fuel_Type -- Chi2 4.102 -- p-value 0.043
Transmission -- Chi2 0.639 -- p-value 0.424
Owner_Type -- Chi2 11.013 -- p-value 0.012
Seats -- K-S nan -- p-value nan
Price -- K-S 0.084 -- p-value 0.495
age -- K-S 0.058 -- p-value 0.894
KM_Driven -- K-S 0.131 -- p-value 0.072
make -- Chi2 13.928 -- p-value 0.455
mileage -- K-S 0.114 -- p-value 0.158
engine -- K-S 0.167 -- p-value 0.009
power -- K-S 0.150 -- p-value 0.026


### Checking the distribution of Owner_Type in training and production data

In [21]:
X_train.Owner_Type.value_counts()

,count
Owner_Type,
First,783
Second,127
Third,24


In [22]:
X_prod.Owner_Type.value_counts()

,count
Owner_Type,
First,84
Second,18
Fourth & Above,1
Third,1
